# 📊 01 — Exploration des Données (EDA)

**Objectif** : Comprendre la structure, la distribution et la qualité des données avant de construire des features et des modèles.

Ce notebook explore le dataset nettoyé `data/interim/cleaned_matches.csv` produit par le pipeline de la Phase 2.

**Questions auxquelles ce notebook répond :**
1. Quelle est la distribution des matchs par ligue, par position, par année ?
2. Comment se comparent les statistiques entre les ligues (LFL vs PRM vs LEC) ?
3. Quelles features sont les plus discriminantes pour la target variable ?
4. Y a-t-il des corrélations fortes entre les stats ?
5. À quoi ressemble le déséquilibre de classes de notre target variable ?

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from IPython.display import display
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # backend non-interactif pour l'exécution en batch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Style global des plots
plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Palette personnalisée pour les ligues
LEAGUE_COLORS = {
    'LEC': '#FFD700',
    'LFL': '#1E90FF',
    'LFL2': '#87CEEB',
    'PRM': '#FF6347',
    'LVP SL': '#FF8C00',
    'NLC': '#9370DB',
    'TCL': '#00CED1',
}

# Créer le dossier pour les figures
Path('../reports/figures').mkdir(parents=True, exist_ok=True)

print('✅ Imports OK')

✅ Imports OK


---
## 1. Chargement des données

In [2]:
# Charger le dataset nettoyé
DATA_PATH = Path('..') / 'data' / 'interim' / 'cleaned_matches.csv'
df = pd.read_csv(DATA_PATH)

# Convertir promoted_to_lec en booléen (le CSV le sauvegarde en string)
if 'promoted_to_lec' in df.columns:
    df['promoted_to_lec'] = df['promoted_to_lec'].astype(bool)

print(f'Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Mémoire : {df.memory_usage(deep=True).sum() / 1e6:.1f} Mo')
df.head()

Dataset chargé : 39,950 lignes × 36 colonnes
Mémoire : 29.1 Mo


,gameid,league,split,date,position,playername,teamname,champion,result,kills,...,damagetochampions,damageshare,dpm,cspm,vspm,earnedgoldshare,gamelength,_source_year,playername_original,promoted_to_lec
0,LOLTMNT06_13630,LEC,Winter,2024-01-13 16:10:20,top,adam,Team BDS,Renekton,0,3,...,10063,0.286116,345.2144,7.9588,0.6861,0.257673,1749,2024,Adam,True
1,LOLTMNT06_13630,LEC,Winter,2024-01-13 16:10:20,jng,sheo,Team BDS,Nocturne,0,2,...,4562,0.129709,156.5009,7.3070,1.4065,0.186460,1749,2024,Sheo,False
2,LOLTMNT06_13630,LEC,Winter,2024-01-13 16:10:20,mid,nuc,Team BDS,Akali,0,2,...,11408,0.324358,391.3551,7.6844,0.7547,0.198391,1749,2024,nuc,False
3,LOLTMNT06_13630,LEC,Winter,2024-01-13 16:10:20,bot,ice,Team BDS,Kalista,0,2,...,6014,0.170993,206.3122,10.0172,0.9605,0.242290,1749,2024,Ice,False
4,LOLTMNT06_13630,LEC,Winter,2024-01-13 16:10:20,sup,labrov,Team BDS,Pyke,0,1,...,3124,0.088823,107.1698,1.5094,5.5575,0.115186,1749,2024,Labrov,False


In [3]:
# Types de données
print('── Types de données ──')
print(df.dtypes.value_counts())
print()
print('── Colonnes ──')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col:25s}  ({df[col].dtype})')

── Types de données ──
int64      13
float64    13
object      9
bool        1
Name: count, dtype: int64

── Colonnes ──
   1. gameid                     (object)
   2. league                     (object)
   3. split                      (object)
   4. date                       (object)
   5. position                   (object)
   6. playername                 (object)
   7. teamname                   (object)
   8. champion                   (object)
   9. result                     (int64)
  10. kills                      (int64)
  11. deaths                     (int64)
  12. assists                    (int64)
  13. killsat15                  (float64)
  14. deathsat15                 (float64)
  15. assistsat15                (float64)
  16. csat15                     (float64)
  17. csdiffat15                 (float64)
  18. golddiffat15               (float64)
  19. xpdiffat15                 (float64)
  20. totalgold                  (int64)
  21. earnedgold                 (int

In [4]:
# Statistiques descriptives des colonnes numériques
df.describe().round(2)

,result,kills,deaths,assists,killsat15,deathsat15,assistsat15,csat15,csdiffat15,golddiffat15,...,wardsplaced,visionscore,damagetochampions,damageshare,dpm,cspm,vspm,earnedgoldshare,gamelength,_source_year
count,39950.0,39950.00,39950.00,39950.00,39950.00,39950.00,39950.00,39950.00,39950.00,39950.00,...,39950.00,39950.00,39950.00,39950.00,39950.00,39950.00,39950.00,39950.00,39950.00,39950.00
mean,0.5,3.08,3.08,7.14,0.82,0.82,1.41,105.72,0.00,0.00,...,21.09,52.48,17350.11,0.20,527.99,6.61,1.59,0.20,1952.16,2024.73
std,0.5,2.89,2.07,4.86,1.09,0.95,1.57,45.64,18.42,835.13,...,20.77,37.51,10374.76,0.09,281.44,3.12,1.05,0.06,333.35,0.67
min,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-137.00,-6247.00,...,0.00,1.00,597.00,0.02,25.05,0.00,0.04,0.01,1094.00,2024.00
25%,0.0,1.00,1.00,3.00,0.00,0.00,0.00,94.00,-9.00,-475.00,...,9.00,28.00,9236.25,0.12,296.75,5.38,0.91,0.16,1716.00,2024.00
50%,0.5,2.00,3.00,6.00,0.00,1.00,1.00,122.00,0.00,0.00,...,13.00,39.00,15803.00,0.20,505.14,7.60,1.20,0.21,1901.00,2025.00
75%,1.0,5.00,4.00,10.00,1.00,1.00,2.00,139.00,9.00,475.00,...,19.00,61.00,23146.75,0.27,711.81,8.90,1.71,0.25,2141.00,2025.00
max,1.0,22.00,13.00,32.00,11.00,9.00,13.00,178.00,137.00,6247.00,...,172.00,342.00,102344.00,0.56,2222.60,13.21,6.92,0.40,3621.00,2026.00


---
## 2. Distribution des matchs

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 2a. Par ligue
league_counts = df['league'].value_counts()
colors = [LEAGUE_COLORS.get(l, '#888888') for l in league_counts.index]
league_counts.plot(kind='barh', ax=axes[0], color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Lignes par ligue')
axes[0].set_xlabel('Nombre de lignes')
for i, v in enumerate(league_counts.values):
    axes[0].text(v + 100, i, f'{v:,}', va='center', fontsize=10)

# 2b. Par position
pos_order = ['top', 'jng', 'mid', 'bot', 'sup']
pos_counts = df['position'].value_counts().reindex(pos_order)
pos_counts.plot(kind='bar', ax=axes[1], color='#4ECDC4', edgecolor='white', linewidth=0.5)
axes[1].set_title('Lignes par position')
axes[1].set_ylabel('Nombre de lignes')
axes[1].tick_params(axis='x', rotation=0)

# 2c. Par année
year_counts = df['_source_year'].value_counts().sort_index()
year_counts.plot(kind='bar', ax=axes[2], color='#F7DC6F', edgecolor='white', linewidth=0.5)
axes[2].set_title('Lignes par année')
axes[2].set_ylabel('Nombre de lignes')
axes[2].tick_params(axis='x', rotation=0)
for i, v in enumerate(year_counts.values):
    axes[2].text(i, v + 200, f'{v:,}', ha='center', fontsize=10)

plt.suptitle('Distribution des données', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/01_data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée')

✅ Figure sauvegardée


C:\Users\bower\AppData\Local\Temp\ipykernel_5340\3072938834.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Nombre de joueurs uniques par ligue
players_per_league = df.groupby('league')['playername'].nunique().sort_values(ascending=False)
print('Joueurs uniques par ligue :')
for league, count in players_per_league.items():
    print(f'  {league:10s} : {count:3d} joueurs')
print(f'\n  Total unique (toutes ligues) : {df["playername"].nunique()}')
print(f'  Note : certains joueurs apparaissent dans plusieurs ligues (transferts)')

Joueurs uniques par ligue :
  NLC        : 318 joueurs
  LFL        : 196 joueurs
  TCL        : 181 joueurs
  PRM        : 155 joueurs
  LVP SL     : 144 joueurs
  LFL2       : 141 joueurs
  LEC        : 109 joueurs

  Total unique (toutes ligues) : 916
  Note : certains joueurs apparaissent dans plusieurs ligues (transferts)


---
## 3. Distributions des stats clés

On regarde les distributions des principales métriques de performance pour comprendre leur forme (normale ? skewed ?) et détecter d'éventuels outliers.

In [7]:
# Stats clés à explorer
key_stats = ['kills', 'deaths', 'assists', 'dpm', 'cspm', 'vspm',
             'golddiffat15', 'csdiffat15', 'xpdiffat15',
             'damageshare', 'earnedgoldshare']

# Garder uniquement les stats disponibles
key_stats = [s for s in key_stats if s in df.columns]

n_cols = 4
n_rows = (len(key_stats) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
axes = axes.flatten()

for i, stat in enumerate(key_stats):
    ax = axes[i]
    df[stat].hist(bins=50, ax=ax, color='#4ECDC4', edgecolor='none', alpha=0.7)
    ax.axvline(df[stat].mean(), color='#FF6B6B', linestyle='--', label=f'mean={df[stat].mean():.1f}')
    ax.axvline(df[stat].median(), color='#FFD93D', linestyle='--', label=f'median={df[stat].median():.1f}')
    ax.set_title(stat, fontsize=12)
    ax.legend(fontsize=8)

# Masquer les axes inutilisés
for j in range(len(key_stats), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution des statistiques clés', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/01_stats_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée')

✅ Figure sauvegardée


C:\Users\bower\AppData\Local\Temp\ipykernel_5340\2166633885.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Comparaison entre ligues

C'est l'analyse la plus importante pour notre outil de scouting. Un joueur qui domine en LFL2 n'a pas le même niveau qu'un joueur moyen en LEC. On doit comprendre ces différences.

In [8]:
# Ordre des ligues par niveau (du plus faible au plus fort)
league_order = ['LFL2', 'NLC', 'PRM', 'LVP SL', 'TCL', 'LFL', 'LEC']
# Garder seulement les ligues présentes
league_order = [l for l in league_order if l in df['league'].unique()]

# Métriques à comparer
compare_stats = ['dpm', 'cspm', 'vspm', 'golddiffat15', 'csdiffat15', 'damageshare']
compare_stats = [s for s in compare_stats if s in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, stat in enumerate(compare_stats):
    ax = axes[i]
    palette = [LEAGUE_COLORS.get(l, '#888888') for l in league_order]
    sns.boxplot(
        data=df, x='league', y=stat, order=league_order,
        hue='league', hue_order=league_order,
        palette=palette, ax=ax, fliersize=1, linewidth=0.8, legend=False
    )
    ax.set_title(stat, fontsize=13)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Comparaison des stats par ligue', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/01_league_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée')

✅ Figure sauvegardée


C:\Users\bower\AppData\Local\Temp\ipykernel_5340\3182676432.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Tableau récapitulatif : médiane de chaque stat par ligue
# La médiane est plus robuste que la moyenne (moins sensible aux outliers)
summary = df.groupby('league')[compare_stats].median().reindex(league_order)
print('Médiane des stats par ligue :')
summary.round(2)

Médiane des stats par ligue :


,dpm,cspm,vspm,golddiffat15,csdiffat15,damageshare
league,,,,,,
LFL2,523.04,7.33,1.08,0.0,0.0,0.20
NLC,532.88,7.41,1.03,0.0,0.0,0.20
PRM,524.08,7.65,1.15,0.0,0.0,0.20
LVP SL,492.10,7.42,1.19,0.0,0.0,0.20
TCL,498.89,7.67,1.21,0.0,0.0,0.21
LFL,505.60,7.76,1.27,0.0,0.0,0.20
LEC,475.37,7.84,1.33,0.0,0.0,0.20


---
## 5. Analyse par position

Les stats brutes ne sont pas comparables entre positions. Un support aura naturellement moins de CS/min qu'un ADC. Notre feature engineering devra en tenir compte.

In [10]:
pos_order = ['top', 'jng', 'mid', 'bot', 'sup']
pos_stats = ['kills', 'deaths', 'assists', 'cspm', 'dpm', 'vspm']
pos_stats = [s for s in pos_stats if s in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

pos_palette = {'top': '#E74C3C', 'jng': '#2ECC71', 'mid': '#3498DB', 'bot': '#F39C12', 'sup': '#9B59B6'}

for i, stat in enumerate(pos_stats):
    ax = axes[i]
    sns.boxplot(
        data=df, x='position', y=stat, order=pos_order,
        hue='position', hue_order=pos_order,
        palette=pos_palette, ax=ax, fliersize=1, linewidth=0.8, legend=False
    )
    ax.set_title(stat, fontsize=13)
    ax.set_xlabel('')

plt.suptitle('Distributions des stats par position', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/01_position_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée')

✅ Figure sauvegardée


C:\Users\bower\AppData\Local\Temp\ipykernel_5340\1376935869.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 💡 Insight
Les distributions varient fortement entre positions → le feature engineering devra normaliser par position (z-score par position par ligue) pour que les comparaisons inter-joueurs soient justes.

---
## 6. Target Variable — Analyse du déséquilibre

Notre target `promoted_to_lec` est binaire : le joueur a-t-il été promu en LEC ? On s'attend à un fort déséquilibre (très peu de joueurs promus).

In [11]:
# Filtrer uniquement les joueurs ERL (les joueurs LEC ne sont pas des candidats)
erl_leagues = ['LFL', 'LFL2', 'PRM', 'LVP SL', 'NLC', 'TCL']
df_erl = df[df['league'].isin(erl_leagues)].copy()

# Balance de la target
target_counts = df_erl['promoted_to_lec'].value_counts()
print('Target Variable (lignes ERL) :')
print(f'  Not promoted : {target_counts.get(False, 0):,} lignes ({target_counts.get(False, 0)/len(df_erl)*100:.1f}%)')
print(f'  Promoted     : {target_counts.get(True, 0):,} lignes ({target_counts.get(True, 0)/len(df_erl)*100:.1f}%)')
print()

# Au niveau joueur (pas ligne)
player_labels = df_erl.groupby('playername')['promoted_to_lec'].max()
promoted_count = int(player_labels.sum())
total_count = len(player_labels)
print(f'Au niveau joueur :')
print(f'  {promoted_count} promus sur {total_count} joueurs ERL ({promoted_count/total_count*100:.1f}%)')
print(f'  Ratio négatif/positif : {(total_count-promoted_count)//max(promoted_count,1)}:1')

Target Variable (lignes ERL) :
  Not promoted : 28,303 lignes (87.4%)
  Promoted     : 4,077 lignes (12.6%)

Au niveau joueur :
  65 promus sur 872 joueurs ERL (7.5%)
  Ratio négatif/positif : 12:1


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart du déséquilibre
labels_pie = ['Non promu', 'Promu en LEC']
sizes = [total_count - promoted_count, promoted_count]
colors_pie = ['#636e72', '#00b894']
explode = (0, 0.08)
axes[0].pie(sizes, explode=explode, labels=labels_pie, colors=colors_pie,
            autopct='%1.1f%%', shadow=True, startangle=140,
            textprops={'fontsize': 12})
axes[0].set_title('Répartition promoted (niveau joueur)', fontsize=13)

# Promus par ligue d'origine
promoted_by_league = (
    df_erl[df_erl['promoted_to_lec'] == True]
    .groupby('league')['playername'].nunique()
    .sort_values(ascending=True)
)
colors_bar = [LEAGUE_COLORS.get(l, '#888888') for l in promoted_by_league.index]
promoted_by_league.plot(kind='barh', ax=axes[1], color=colors_bar, edgecolor='white')
axes[1].set_title("Joueurs promus par ligue d'origine", fontsize=13)
axes[1].set_xlabel('Nombre de joueurs promus')
for i, v in enumerate(promoted_by_league.values):
    axes[1].text(v + 0.3, i, str(v), va='center', fontsize=11)

plt.tight_layout()
plt.savefig('../reports/figures/01_target_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée')

✅ Figure sauvegardée


C:\Users\bower\AppData\Local\Temp\ipykernel_5340\1186443064.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 💡 Insight
Le dataset est fortement déséquilibré (~13:1). On devra utiliser des techniques de gestion du déséquilibre :
- `class_weight='balanced'` dans les modèles Sklearn
- `scale_pos_weight` dans XGBoost
- Métriques adaptées : AUC-ROC, Precision-Recall plutôt qu'accuracy

---
## 7. Stats des joueurs promus vs non-promus

La question clé : les joueurs qui ont été promus en LEC avaient-ils des stats significativement différentes quand ils étaient en ERL ?

In [13]:
# Comparer les stats moyennes des promus vs non-promus (ERL uniquement)
compare_cols = ['kills', 'deaths', 'assists', 'dpm', 'cspm', 'vspm',
                'golddiffat15', 'csdiffat15', 'xpdiffat15',
                'damageshare', 'earnedgoldshare']
compare_cols = [c for c in compare_cols if c in df_erl.columns]

comparison = df_erl.groupby('promoted_to_lec')[compare_cols].mean()
comparison.index = ['Non promu', 'Promu']

# Calculer le % de différence
diff_pct = ((comparison.loc['Promu'] - comparison.loc['Non promu']) / comparison.loc['Non promu'].replace(0, np.nan) * 100)

print('Comparaison Promus vs Non-Promus (moyennes ERL) :')
print('=' * 60)
for col in compare_cols:
    non_p = comparison.loc['Non promu', col]
    pro = comparison.loc['Promu', col]
    diff = diff_pct[col]
    arrow = '↑' if diff > 0 else '↓'
    print(f'  {col:20s}  Non-promu: {non_p:8.2f}  Promu: {pro:8.2f}  ({arrow} {abs(diff):.1f}%)')

Comparaison Promus vs Non-Promus (moyennes ERL) :
  kills                 Non-promu:     3.15  Promu:     3.27  (↑ 3.9%)
  deaths                Non-promu:     3.24  Promu:     2.67  (↓ 17.5%)
  assists               Non-promu:     7.24  Promu:     7.65  (↑ 5.6%)
  dpm                   Non-promu:   533.37  Promu:   541.53  (↑ 1.5%)
  cspm                  Non-promu:     6.54  Promu:     6.76  (↑ 3.3%)
  vspm                  Non-promu:     1.53  Promu:     1.67  (↑ 8.9%)
  golddiffat15          Non-promu:   -21.61  Promu:   150.04  (↓ 794.2%)
  csdiffat15            Non-promu:    -0.46  Promu:     3.19  (↓ 794.2%)
  xpdiffat15            Non-promu:   -18.14  Promu:   125.95  (↓ 794.2%)
  damageshare           Non-promu:     0.20  Promu:     0.20  (↓ 1.1%)
  earnedgoldshare       Non-promu:     0.20  Promu:     0.20  (↑ 0.0%)


In [14]:
# Visualisation : violinplots des stats promus vs non-promus
viz_stats = ['dpm', 'cspm', 'golddiffat15', 'csdiffat15']
viz_stats = [s for s in viz_stats if s in df_erl.columns]

# Créer une colonne string pour le hue (évite les problèmes bool/seaborn)
df_erl['promoted_label'] = df_erl['promoted_to_lec'].map({True: 'Promu', False: 'Non promu'})

fig, axes = plt.subplots(1, len(viz_stats), figsize=(5*len(viz_stats), 6))
if len(viz_stats) == 1:
    axes = [axes]

for i, stat in enumerate(viz_stats):
    ax = axes[i]
    sns.violinplot(
        data=df_erl, x='promoted_label', y=stat,
        order=['Non promu', 'Promu'],
        hue='promoted_label', hue_order=['Non promu', 'Promu'],
        palette={'Non promu': '#636e72', 'Promu': '#00b894'},
        ax=ax, inner='quartile', linewidth=0.8, legend=False
    )
    ax.set_title(stat, fontsize=13)
    ax.set_xlabel('')

plt.suptitle('Promus vs Non-Promus (ERL)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/01_promoted_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée')

✅ Figure sauvegardée


C:\Users\bower\AppData\Local\Temp\ipykernel_5340\83782511.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 8. Matrice de corrélation

On cherche les features redondantes (corrélation > 0.9) et les features les plus corrélées avec la target.

In [15]:
# Sélectionner les colonnes numériques
numeric_cols = df_erl.select_dtypes(include=[np.number]).columns.tolist()
# Exclure les colonnes non-pertinentes
exclude = ['_source_year', 'gamelength']
numeric_cols = [c for c in numeric_cols if c not in exclude]

corr_matrix = df_erl[numeric_cols].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Triangle supérieur
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 8}
)
ax.set_title('Matrice de corrélation des features numériques', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/01_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée')

✅ Figure sauvegardée


C:\Users\bower\AppData\Local\Temp\ipykernel_5340\3370854809.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Features les plus corrélées avec la target
if 'promoted_to_lec' in corr_matrix.columns:
    target_corr = corr_matrix['promoted_to_lec'].drop('promoted_to_lec').abs().sort_values(ascending=False)
    print('Top features corrélées avec promoted_to_lec :')
    print('=' * 50)
    for feat, corr in target_corr.head(15).items():
        direction = '+' if corr_matrix.loc[feat, 'promoted_to_lec'] > 0 else '-'
        print(f'  {feat:25s} : {direction}{corr:.3f}')

In [17]:
# Features hautement corrélées entre elles (> 0.9)
# Ce sont des candidats pour suppression (redondance)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print(f'\nPaires de features très corrélées (|r| > 0.9) :')
    for f1, f2, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
        print(f'  {f1:25s} ↔ {f2:25s} : r = {r:.3f}')
else:
    print('Aucune paire avec |r| > 0.9')


Paires de features très corrélées (|r| > 0.9) :
  totalgold                 ↔ earnedgold                : r = 0.984
  csat15                    ↔ cspm                      : r = 0.961
  visionscore               ↔ vspm                      : r = 0.958
  damagetochampions         ↔ dpm                       : r = 0.941
  wardsplaced               ↔ visionscore               : r = 0.936
  wardsplaced               ↔ vspm                      : r = 0.912
  cspm                      ↔ earnedgoldshare           : r = 0.910


---
## 9. Analyse des champions

Exploration rapide de la diversité de champions pour potentiellement créer des features de champion pool plus tard.

In [18]:
# Champion pool par joueur
champ_pool = df_erl.groupby('playername')['champion'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution de la taille du champion pool
champ_pool.hist(bins=30, ax=axes[0], color='#6C5CE7', edgecolor='none', alpha=0.7)
axes[0].set_title('Taille du champion pool par joueur')
axes[0].set_xlabel('Nombre de champions uniques joués')
axes[0].set_ylabel('Nombre de joueurs')
axes[0].axvline(champ_pool.mean(), color='#FF6B6B', linestyle='--', label=f'Moyenne: {champ_pool.mean():.1f}')
axes[0].legend()

# Champion pool promus vs non-promus
player_promoted = df_erl.groupby('playername')['promoted_to_lec'].max()
champ_pool_df = pd.DataFrame({'champion_pool': champ_pool, 'promoted': player_promoted})
champ_pool_df['promoted_label'] = champ_pool_df['promoted'].map({True: 'Promu', False: 'Non promu'})
sns.boxplot(
    data=champ_pool_df, x='promoted_label', y='champion_pool',
    order=['Non promu', 'Promu'],
    hue='promoted_label', hue_order=['Non promu', 'Promu'],
    palette={'Non promu': '#636e72', 'Promu': '#00b894'},
    ax=axes[1], linewidth=0.8, legend=False
)
axes[1].set_title('Champion pool : Promus vs Non-Promus')
axes[1].set_xlabel('')
axes[1].set_ylabel('Champions uniques joués')

plt.tight_layout()
plt.savefig('../reports/figures/01_champion_pool.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée')

✅ Figure sauvegardée


C:\Users\bower\AppData\Local\Temp\ipykernel_5340\4114693850.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 10. Résumé et insights clés

### Ce qu'on retient pour la suite

1. **Déséquilibre de classes** : ~7-8% seulement des joueurs ERL sont promus. Il faudra des techniques adaptées (class_weight, SMOTE, Precision-Recall).

2. **Différences inter-ligues** : Les stats brutes ne sont pas comparables entre ligues. Un bon joueur de LFL2 peut avoir des stats inférieures à un joueur moyen de LEC. → Le feature engineering devra inclure des **z-scores par ligue**.

3. **Différences par position** : Les supports ont naturellement moins de CS, DPM que les carries. → Normaliser aussi **par position**.

4. **Features prometteuses** : Les stats d'early game (golddiffat15, csdiffat15) et d'efficacité (dpm, cspm) semblent discriminantes entre promus et non-promus.

5. **Corrélations fortes** : Certaines features sont redondantes (kills ↔ assists, totalgold ↔ earnedgold). On pourra en supprimer ou utiliser PCA.

6. **Champion pool** : Les promus tendent à avoir un champion pool plus large, ce qui est une feature intéressante à ajouter.

### Prochaine étape
→ **Phase 4 : Feature Engineering** — Construire des features agrégées par joueur par split, avec normalisation par ligue et par position.